In [1]:
# Get the sqlite database path
#| echo: false

from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text, bindparam

from nhs_waiting_lists import (
    __app_name__,
)
from nhs_waiting_lists.constants import LARGE_ACUTE_PROVIDER_CODES, TOTAL_ONLY_TREATMENT_CODES
from nhs_waiting_lists.constants import proj_db_path, DB_FILE
from nhs_waiting_lists.utils.xdg import XDGBasedir

project_root = Path(XDGBasedir.get_data_dir(__app_name__))

DB_PATH = project_root / proj_db_path / DB_FILE
FILES_DIR = project_root / "files"

engine = create_engine(f"sqlite:///{DB_PATH}")

# Get the 23 large acute trust provider codes, identified by the ranking table csv
PROVIDER_CODES = LARGE_ACUTE_PROVIDER_CODES

# Get the C_999 meta-treatment code which aggregates all other treatment codes
TREATMENT_CODES = TOTAL_ONLY_TREATMENT_CODES
# TREATMENT_CODES = ALL_TREATMENT_CODES

#
# provider_code = "RAJ"
# treatment_code = "C_110"

In [2]:
#| echo: false
#| output: false

## Retrieve all the provider rtt summary data

consolidated_query = text("""
                          SELECT i.period,
                                 i.provider,
                                 i.treatment,
                                 i.untreated       AS untreated,
                                 i.new_periods     AS new_periods,
                                 i.incomplete,
                                 i.incomplete_prev AS incomplete_prev,
                                 i.treated         AS treated,
                              p.provider_name
                          FROM consolidated AS i
                                   JOIN provider AS p ON i.provider = p.provider
                          WHERE i.provider IN :provider_codes
                            AND i.treatment IN :treatment_codes
                            AND i.period >= '2024-01'
                            AND i.period <= '2024-12'
                            AND subtype = 'Acute - Large'
                          ORDER BY i.provider ASC, i.period ASC; \
                          """).bindparams(
    bindparam('provider_codes', expanding=True),
    bindparam('treatment_codes', expanding=True)
)

consolidated_df = pd.read_sql(consolidated_query, engine, params={
    'provider_codes': PROVIDER_CODES,
    'treatment_codes': TREATMENT_CODES}
                              )  # type: ignore[arg-type]

consolidated_df["period"] = pd.to_datetime(consolidated_df["period"], errors="coerce")

# valid = (consolidated_df['incomplete_prev'] > 100) & (consolidated_df['new_periods'] > 30)
# consolidated_df = consolidated_df.loc[valid].copy()



In [3]:
# Echo the base df before applying style sheets
#| echo: false
#| output: false

other_result: pd.DataFrame = consolidated_df.groupby(['period', 'provider'])[
    ['incomplete_prev', 'new_periods', 'treated', 'untreated', "incomplete"]].sum().reset_index()

other_result["total_pathways"] = other_result["incomplete_prev"] + other_result["new_periods"]

other_result["unexplained"] = other_result["incomplete"] - (
        other_result["incomplete_prev"] + other_result["new_periods"] - other_result["treated"])

other_result["unexplained_pct"] = other_result["untreated"] / other_result["total_pathways"]

other_result["expected_pathways"] = other_result["incomplete_prev"] + other_result["new_periods"] - other_result[
    "treated"]

other_result["actual_pathways"] = other_result["incomplete"]

df_sorted = other_result.sort_values(by=['unexplained_pct'])[["period", "provider", "unexplained_pct", "unexplained"]]
df_top_20 = df_sorted.head(10)
df_final = df_top_20.reset_index(drop=True)
df_final

,period,provider,unexplained_pct,unexplained
0,2024-04-01,RHW,-0.160402,-6998
1,2024-01-01,RHW,-0.157235,-6899
2,2024-10-01,RHW,-0.153375,-6941
3,2024-07-01,RHW,-0.141166,-6095
4,2024-03-01,RHW,-0.127510,-5665
5,2024-12-01,RHW,-0.121442,-5432
6,2024-05-01,RHW,-0.119391,-4994
7,2024-02-01,RHW,-0.113517,-4955
8,2024-06-01,RHW,-0.113284,-4779
9,2024-07-01,RAJ,-0.106382,-22639


In [4]:
other_result.sort_values(by=['unexplained'])[["period", "provider", "unexplained_pct", "unexplained"]].head(20).reset_index(drop=True)

,period,provider,unexplained_pct,unexplained
0,2024-07-01,RAJ,-0.106382,-22639
1,2024-03-01,RAJ,-0.101640,-21245
2,2024-10-01,RAJ,-0.094658,-19801
3,2024-01-01,RAJ,-0.094100,-19284
4,2024-11-01,RAJ,-0.092199,-19120
5,2024-09-01,RAJ,-0.087315,-18304
6,2024-05-01,RAJ,-0.085040,-17804
7,2024-12-01,RAJ,-0.084992,-17062
8,2024-06-01,RAJ,-0.077418,-16160
9,2024-02-01,RAJ,-0.078927,-16091


In [5]:
all_provider_rankings: pd.DataFrame = consolidated_df.groupby(['provider','provider_name'])[
    ['incomplete_prev', 'new_periods', 'treated', 'untreated', "incomplete"]].sum().reset_index()

all_provider_rankings["total_pathways"] = all_provider_rankings["incomplete_prev"] + all_provider_rankings["new_periods"]

all_provider_rankings["unexplained"] = all_provider_rankings["incomplete"] - (
        all_provider_rankings["incomplete_prev"] + all_provider_rankings["new_periods"] - all_provider_rankings["treated"])

all_provider_rankings["unexplained_pct"] = all_provider_rankings["untreated"] / all_provider_rankings["total_pathways"]

all_provider_rankings["expected_pathways"] = all_provider_rankings["incomplete_prev"] + all_provider_rankings["new_periods"] - all_provider_rankings[
    "treated"]

all_provider_rankings["actual_pathways"] = all_provider_rankings["incomplete"]


all_provider_rankings.sort_values(by=['unexplained'])[["provider",'provider_name', "unexplained_pct", "unexplained"]].head(10).reset_index(drop=True)

,provider,provider_name,unexplained_pct,unexplained
0,RAJ,Mid and South Essex NHS Foundation Trust,-0.086708,-215408
1,RTE,Gloucestershire Hospitals NHS Foundation Trust,-0.079950,-88038
2,RHW,Royal Berkshire NHS Foundation Trust,-0.125333,-65209
3,RJ2,Lewisham and Greenwich NHS Trust,-0.044870,-44480
4,RXR,East Lancashire Hospitals NHS Trust,-0.041111,-41632
5,R0B,South Tyneside and Sunderland NHS Foundation T...,-0.037242,-35621
6,RDE,East Suffolk and North Essex NHS Foundation Trust,-0.025792,-33424
7,RWH,East and North Hertfordshire NHS Trust,-0.036438,-30656
8,REF,Royal Cornwall Hospitals NHS Trust,-0.042675,-29250
9,RTF,Northumbria Healthcare NHS Foundation Trust,-0.054647,-28307


In [6]:
all_provider_rankings.sort_values(by=['unexplained_pct'])[["provider",'provider_name', "unexplained_pct", "unexplained"]].head(10).reset_index(drop=True)

,provider,provider_name,unexplained_pct,unexplained
0,RHW,Royal Berkshire NHS Foundation Trust,-0.125333,-65209
1,RAJ,Mid and South Essex NHS Foundation Trust,-0.086708,-215408
2,RTE,Gloucestershire Hospitals NHS Foundation Trust,-0.079950,-88038
3,RTF,Northumbria Healthcare NHS Foundation Trust,-0.054647,-28307
4,RJ2,Lewisham and Greenwich NHS Trust,-0.044870,-44480
5,REF,Royal Cornwall Hospitals NHS Trust,-0.042675,-29250
6,RXR,East Lancashire Hospitals NHS Trust,-0.041111,-41632
7,R0B,South Tyneside and Sunderland NHS Foundation T...,-0.037242,-35621
8,RWH,East and North Hertfordshire NHS Trust,-0.036438,-30656
9,RVJ,North Bristol NHS Trust,-0.033285,-23393


In [7]:
(df_final.style.format(precision=2, thousands=",", decimal=".")
 .format('{:.2%}', subset=["unexplained_pct"])
 .format(lambda v: v.strftime("%Y-%m"), subset=["period"])
 )

,period,provider,unexplained_pct,unexplained
0,2024-04,RHW,-16.04%,"-6,998"
1,2024-01,RHW,-15.72%,"-6,899"
2,2024-10,RHW,-15.34%,"-6,941"
3,2024-07,RHW,-14.12%,"-6,095"
4,2024-03,RHW,-12.75%,"-5,665"
5,2024-12,RHW,-12.14%,"-5,432"
6,2024-05,RHW,-11.94%,"-4,994"
7,2024-02,RHW,-11.35%,"-4,955"
8,2024-06,RHW,-11.33%,"-4,779"
9,2024-07,RAJ,-10.64%,"-22,639"


In [8]:
#| echo: false
#| output: true

(other_result[
     [
         "period",
         "actual_pathways",
         "expected_pathways",
         "unexplained",
         "total_pathways",
         "unexplained_pct",
     ]]
 .style.relabel_index(
    [
        "",
        'Observed<br>Incomplete<br>Pathways',
        'Expected<br>Incomplete<br>Pathways',
        'Unreported<br>Change',
        'Treatable<br>Pathways',
        'Unreported<br>%'
    ], axis=1).hide(axis='index')
 .format(precision=3, thousands=",", decimal=".")
 .format('{:.2%}', subset=["unexplained_pct"])
 .format(lambda v: v.strftime("%Y-%m"), subset=["period"])
 )

,ObservedIncompletePathways,ExpectedIncompletePathways,UnreportedChange,TreatablePathways,Unreported%
2024-01,"60,893","63,764","-2,871","79,362",-3.62%
2024-01,"160,406","179,690","-19,284","204,932",-9.41%
2024-01,"87,455","89,992","-2,537","106,587",-2.38%
2024-01,"80,905","82,765","-1,860","96,056",-1.94%
2024-01,"43,406","45,797","-2,391","58,565",-4.08%
2024-01,"81,599","83,119","-1,520","97,320",-1.56%
2024-01,"74,539","76,168","-1,629","92,556",-1.76%
2024-01,"59,369","61,380","-2,011","73,543",-2.73%
2024-01,"32,034","38,933","-6,899","43,877",-15.72%
2024-01,"67,679","72,087","-4,408","83,362",-5.29%
